# 03 — Definición reproducible de la cohorte adulta

Aplica la lógica validada de `mimic_sepsis.cohort` al demo MIMIC-IV v2.2. La política primaria provisional conserva la primera estancia UCI elegible de cada ingreso hospitalario. Se comparan alternativas antes de congelar la decisión para el análisis completo.

**Unidad fuente:** estancia UCI. **Edad mínima:** 18 años. **Salida visible:** exclusivamente recuentos agregados.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from mimic_sepsis.cohort import StayPolicy, build_adult_icu_cohort

DATA_DIR = PROJECT_ROOT / 'data' / 'mimic-iv-demo' / '2.2'

In [ ]:
def read_demo_table(domain: str, table: str) -> pd.DataFrame:
    path = DATA_DIR / domain / f'{table}.csv.gz'
    if not path.exists():
        raise FileNotFoundError(f'Falta {path}; ejecute primero el notebook 00')
    return pd.read_csv(path, low_memory=False)

patients = read_demo_table('hosp', 'patients')
admissions = read_demo_table('hosp', 'admissions')
icustays = read_demo_table('icu', 'icustays')
print(f'Tablas cargadas: {len(patients)} pacientes, {len(admissions)} ingresos, {len(icustays)} estancias UCI')

## Cohorte primaria provisional

In [ ]:
primary = build_adult_icu_cohort(
    patients, admissions, icustays,
    minimum_age=18,
    stay_policy=StayPolicy.FIRST_PER_ADMISSION,
)
pd.DataFrame([{'stage': key, 'count': value} for key, value in primary.flow.items()])

In [ ]:
exclusions = (
    primary.audit.loc[~primary.audit['selected'], 'exclusion_reason']
    .fillna('unspecified')
    .value_counts()
    .rename_axis('exclusion_reason')
    .reset_index(name='icu_stays')
)
exclusions

## Sensibilidad a la política de estancias

In [ ]:
policy_comparison = []
for policy in StayPolicy:
    result = build_adult_icu_cohort(
        patients, admissions, icustays, minimum_age=18, stay_policy=policy
    )
    policy_comparison.append({
        'policy': policy.value,
        'icu_stays': len(result.cohort),
        'patients': result.cohort['subject_id'].nunique(),
        'admissions': result.cohort['hadm_id'].nunique(),
    })
pd.DataFrame(policy_comparison)

## Invariantes de la cohorte seleccionada

In [ ]:
cohort = primary.cohort
assert cohort['stay_id'].is_unique
assert not cohort.duplicated(['subject_id', 'hadm_id']).any()
assert cohort['age_at_icu'].ge(18).all()
assert cohort['outtime'].gt(cohort['intime']).all()
assert cohort[['subject_id', 'hadm_id', 'stay_id']].notna().all().all()
print('Invariantes superados: unicidad, edad, cronología e identificadores.')

## Decisión provisional

Se mantiene `first_per_admission` como análisis primario porque evita contar repetidamente traslados/reingresos dentro del mismo ingreso y conserva nuevos ingresos del mismo paciente. La división de desarrollo/validación seguirá agrupada por `subject_id`. Se preespecifican `first_per_patient` y `all` como sensibilidades. Esta decisión deberá confirmarse con los recuentos de la versión completa antes de bloquear el test.